# 모듈 (3/5): Graph RAG — Neo4j · LlamaIndex GraphRAG
## 지식(Memory & Knowledge) 시리즈

---

[2편](M04_2_vector_rag.ipynb)의 Vector RAG 는 "**질문과 비슷한** 청크"를 찾습니다.
하지만 답이 **여러 문서에 흩어져 관계로만 이어져 있을 때** 는 유사도만으로는 닿지 못합니다.
그 빈틈을 **지식 그래프** 가 메웁니다.

### 학습 목표
1. **지식 그래프 기본** — 엔티티·관계·방향(direction)·multi-hop 탐색 (`rag.KnowledgeGraph`)
2. **Neo4j 벡터 인덱스** — 하나의 DB 에서 벡터 검색 + 그래프 확장 (`rag.Neo4jVectorStore`)
3. **LlamaIndex RAG** — LangChain 과 다른 또 하나의 RAG 프레임워크
4. **GraphRAG** — LLM 이 지식 그래프를 **자동 추출**하고, **커뮤니티 요약**으로
   전역(Global) 질문까지 답한다 (`PropertyGraphIndex` + Microsoft GraphRAG 아이디어)
5. **Hybrid RAG** — 벡터 검색 결과 + 그래프 컨텍스트를 합쳐 답변
6. **그래프 DB 갈아타기** — Memgraph 로 같은 Cypher 를 그대로 재사용
7. **Text2Cypher** — 자연어 질문을 Cypher 로 번역 (LangChain `GraphCypherQAChain`)

### 이 시리즈 구성
1. `M04_1_embeddings.ipynb` — 임베딩 · 시맨틱 유사도 · 공급자 비교
2. `M04_2_vector_rag.ipynb` — 문서 전처리 · 벡터 DB 3종 · LangChain RAG
3. **`M04_3_graph_rag.ipynb`** ← (현재) 지식 그래프 · Neo4j · LlamaIndex GraphRAG
4. `M04_4_agent_memory.ipynb` — 에이전트 메모리 **직접 구현** · Deep-Knowledge Agent
5. `M04_5_memory.ipynb` — LangGraph **메모리 인프라**(체크포인터 · Store · 영속화)

> 📦 **환경 설치·실행 명령**은 [`env_guides/M04_3_graph_rag.md`](env_guides/M04_3_graph_rag.md) 참고.
> 구현은 [`agentic_lib/rag.py`](agentic_lib/rag.py)(`KnowledgeGraph`·`Neo4jVectorStore`·`HybridRAG`),
> [`agentic_lib/llama_rag.py`](agentic_lib/llama_rag.py)(LlamaIndex 어댑터),
> [`agentic_lib/graph_stack.py`](agentic_lib/graph_stack.py)(커뮤니티 요약 · 다른 그래프 DB · Text2Cypher)에 있습니다.
> **2편과 같은 개인 문서** 를 그대로 씁니다 — 같은 데이터에서 무엇이 달라지는지 보기 위함입니다.

### Vector RAG 와 Graph RAG 는 무엇이 다른가

```
질문: "회의에서 정한 보안 관련 결정이, 실제 보안 지침의 어떤 조항과 이어지나?"

[Vector RAG]                          [Graph RAG]
질문 임베딩                            질문 → 엔티티 추출("보안", "결정")
   ↓ 코사인 유사도                        ↓ 그래프 탐색
"비슷한 표현"의 청크 k개                 (회의록)-[:MENTIONS]->(보안)<-[:MENTIONS]-(보안지침)
   ↓                                     ↓ 관계를 타고 확장
표현이 다르면 놓친다                    표현이 달라도 '연결'되어 있으면 찾는다
```

| | Vector RAG | Graph RAG |
|---|---|---|
| 찾는 기준 | 의미적 **유사성** | 명시적 **연결성** |
| 잘하는 질문 | "X 가 뭐야?" | "X 와 Y 는 어떤 관계야?", "X 에서 출발해 Z 까지" |
| 준비 비용 | 낮음(청킹+임베딩) | **높음**(엔티티·관계 추출 필요) |
| 약점 | 다단계 추론 불가 | 그래프 품질 = 추출 LLM 품질 |

**결론부터**: 둘은 대체재가 아니라 보완재입니다. 5장의 Hybrid RAG 가 그 조합입니다.

---
## 0. 준비

### 필요 패키지 (CMD)
```bat
REM 벡터 검색 + 임베딩
uv pip install chromadb sentence-transformers numpy

REM Neo4j 드라이버
uv pip install neo4j

REM LlamaIndex (GraphRAG)
uv pip install llama-index-core llama-index-graph-stores-neo4j
```

### Neo4j 실행 (CMD + Docker Desktop)
```bat
REM APOC 플러그인 포함 — LlamaIndex 의 Neo4jPropertyGraphStore 가 apoc.meta.data 를 요구한다
docker run -d --name neo4j -p 7474:7474 -p 7687:7687 ^
  -e NEO4J_AUTH=neo4j/password -e "NEO4J_PLUGINS=[\"apoc\"]" neo4j:5-community
```

> **Neo4j 없이도 1장·3장은 실행됩니다.** 미연결 시 2·4장은 안내 후 건너뜁니다.

> ⚠️ **이 노트북의 답변 품질은 `.env` 의 `LLM_PROVIDER` 에 크게 좌우됩니다.**
> 특히 4장의 GraphRAG 는 LLM 이 트리플을 추출하므로 모델이 약하면 그래프 자체가 부실해집니다.
> `openrouter/free` 처럼 **요청마다 다른 무료 모델로 라우팅되는 설정** 은 실행할 때마다
> 결과가 달라지고, 가끔 `User Safety: safe` 같은 무의미한 응답이 섞이기도 합니다.
> 안정적인 결과를 보려면 `ollama`(qwen3:8b) 나 `google` 처럼 **모델이 고정된 공급자** 를 쓰세요.

In [1]:
# 자기완결 setup: 이 노트북만 단독 실행해도 되도록 공통 셋업을 맨 앞에 둔다
import sys, os
sys.path.insert(0, os.path.abspath(''))  # notebooks/ 를 import 경로에 추가

import utils
utils.reload_env()  # .env 재로드 (LLM_PROVIDER 등 갱신) + 현재 공급자 상태 출력

from utils import uv_install, get_llm, test_llm_connection, LLM_PROVIDER
# 공통 라이브러리
from agentic_lib import bootstrap, embeddings as emb   # 임베딩 공급자(M04_1)
from agentic_lib import doc_prep                       # 개인 문서 구조화(M04_2)
from agentic_lib import rag                            # KnowledgeGraph·Neo4jVectorStore·HybridRAG
from agentic_lib import llama_rag                      # LlamaIndex 어댑터
from agentic_lib import vector_stores as vs            # Chroma/FAISS/Qdrant
from agentic_lib import graph_stack as gstack          # 다른 그래프 DB·Text2Cypher·커뮤니티 요약(6~8장)
from agentic_lib.bootstrap import to_text            # 공급자 무관 응답 정규화(<think>·list content 제거)

uv_install(['chromadb', 'sentence-transformers', 'numpy', 'neo4j',
            'llama-index-core', 'llama-index-graph-stores-neo4j'])

llm = test_llm_connection()  # 공급자 무관
embedder = emb.get_embedder("local")
print(f"임베더: {embedder.name} ({embedder.dim}차원)")

LLM 공급자: openrouter
  OpenRouter Key: 설정됨  /  Model: nvidia/nemotron-3-super-120b-a12b:free


[uv] 설치 완료: ['chromadb', 'sentence-transformers', 'numpy', 'neo4j', 'llama-index-core', 'llama-index-graph-stores-neo4j']


LLM 연결 성공 [openrouter]: 1+1을 계산하면 2입니다.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

임베더: local:paraphrase-multilingual-MiniLM-L12-v2 (384차원)


In [2]:
# 2편과 같은 개인 문서를 준비한다(없으면 생성)
DOC_DIR = os.path.join("workspace", "personal_docs")
doc_prep.write_sample_docs(DOC_DIR)
docs = doc_prep.load_markdown_docs(DOC_DIR)

doc_prep.print_doc_structure(docs)

문서 ID                              분류       작성일            글자수   섹션  태그
------------------------------------------------------------------------------------------------------------
2026-03-14_회의록_프로젝트킥오프             회의록      2026-03-14     439    4  프로젝트, 킥오프, RAG, 보안
건강_러닝기록_3월                         개인메모     2026-03-31     233    3  운동, 러닝, 건강
규정_문서보안_지침                         사내규정     2025-11-05     259    3  보안, 규정, 권한
업무_주간보고_2026-W12                   업무보고     2026-03-20     328    3  프로젝트, 주간보고, RAG
여행_오사카_준비메모                        개인메모     2026-01-11     259    3  여행, 오사카, 체크리스트
학습노트_langchain_lcel                학습노트     2026-02-28     516    4  LangChain, LCEL, RAG
------------------------------------------------------------------------------------------------------------
총 6개 문서 / 20개 섹션 / 2,034자


---
## 1. 지식 그래프 기본 — 순수 Python 으로 원리 보기

DB 없이 dict 두 개(노드·엣지)로 그래프를 만들어 봅니다.
Neo4j 를 쓰기 전에 **무엇이 그래프인지** 를 손으로 확인하는 단계입니다.

```
(문서) --[:MENTIONS]--> (태그)
(태그) --[관계]--> (태그)
```

우리 개인 문서의 `tags` 를 그대로 엔티티로 삼으면 자연스러운 그래프가 나옵니다.

In [3]:
# 개인 문서의 태그를 엔티티로 삼아 지식 그래프를 구성한다
kg = rag.KnowledgeGraph()

for doc in docs:
    kg.add_node(doc.doc_id, "Document", description=doc.title, category=doc.category)
    for tag in doc.tags:
        kg.add_node(tag, "Topic", description=f"태그: {tag}")
        kg.add_edge(doc.doc_id, "mentions", tag)

# 태그 사이의 관계는 '설계'해야 한다 — 자동으로 생기지 않는다
TOPIC_LINKS = [
    ("RAG", "needs", "LangChain"),
    ("킥오프", "part_of", "프로젝트"),
    ("주간보고", "reports", "프로젝트"),
    ("보안", "constrains", "프로젝트"),
    ("권한", "part_of", "보안"),
    ("규정", "defines", "보안"),
]
for source, relation, target in TOPIC_LINKS:
    kg.add_node(source, "Topic"); kg.add_node(target, "Topic")
    kg.add_edge(source, relation, target)

print(f"노드 {len(kg.nodes)}개 / 엣지 {len(kg.edges)}개\n")

# 엣지에는 방향이 있다: from -[relation]-> to
# '프로젝트' 는 (문서 -[mentions]-> 프로젝트, 킥오프 -[part_of]-> 프로젝트 …) 처럼 늘 화살표를 '받는' 쪽이라
# 나가는 엣지만 보는 기본값 direction="out" 으로는 이웃이 0건이다 → "both" 로 양방향을 봐야 한다.
print("=== '프로젝트' 태그의 이웃 ===")
print(f"  direction='out'  → {len(kg.query_neighbors('프로젝트'))}건 (프로젝트는 화살표를 받기만 한다)")
print(f"  direction='both' → {len(kg.query_neighbors('프로젝트', direction='both'))}건")
for neighbor in kg.query_neighbors("프로젝트", direction="both"):
    # direction 이 'out' 이면 내가 가리키는 쪽, 'in' 이면 나를 가리키는 쪽이므로 화살표를 뒤집는다
    arrow = (f"-[{neighbor['relation']}]->" if neighbor["direction"] == "out"
             else f"<-[{neighbor['relation']}]-")
    print(f"    프로젝트 {arrow} {neighbor['node']}")

# 홉(hop) = 엣지를 한 번 건너는 것. 1홉은 직접 이웃, 2홉은 '이웃의 이웃'이다.
print("\n=== Multi-hop: '권한' 에서 2홉 안에 닿는 것 (화살표 방향대로) ===")
for path in kg.multi_hop_query("권한", max_hops=2):
    print(f"  [{len(path)}홉] 권한 " + " ".join(f"-[{rel}]-> {node}" for rel, node in path))

# 방향을 무시하면(= 2장 Cypher 의 무방향 `-[*1..2]-`) 문서 노드까지 끌려 들어와 훨씬 넓어진다
print("\n=== 같은 2홉을 방향 무시(direction='both')로 ===")
for path in kg.multi_hop_query("권한", max_hops=2, direction="both"):
    print(f"  [{len(path)}홉] 권한 " + " ".join(f"~[{rel}]~ {node}" for rel, node in path))

노드 21개 / 엣지 25개

=== '프로젝트' 태그의 이웃 ===
  direction='out'  → 0건 (프로젝트는 화살표를 받기만 한다)
  direction='both' → 5건
    프로젝트 <-[mentions]- 2026-03-14_회의록_프로젝트킥오프
    프로젝트 <-[mentions]- 업무_주간보고_2026-W12
    프로젝트 <-[part_of]- 킥오프
    프로젝트 <-[reports]- 주간보고
    프로젝트 <-[constrains]- 보안

=== Multi-hop: '권한' 에서 2홉 안에 닿는 것 (화살표 방향대로) ===
  [1홉] 권한 -[part_of]-> 보안
  [2홉] 권한 -[part_of]-> 보안 -[constrains]-> 프로젝트

=== 같은 2홉을 방향 무시(direction='both')로 ===
  [1홉] 권한 ~[mentions]~ 규정_문서보안_지침
  [1홉] 권한 ~[part_of]~ 보안
  [2홉] 권한 ~[mentions]~ 규정_문서보안_지침 ~[mentions]~ 규정
  [2홉] 권한 ~[part_of]~ 보안 ~[mentions]~ 2026-03-14_회의록_프로젝트킥오프
  [2홉] 권한 ~[part_of]~ 보안 ~[constrains]~ 프로젝트


### 방향(direction)과 홉(hop) — 그래프를 읽는 두 개의 눈금

**① 방향** — 엣지는 `from -[relation]-> to` 로 한쪽을 향합니다. 그래서 "이웃"에도 세 종류가 있습니다.

| `direction` | 보는 엣지 | `프로젝트` 기준 결과 |
|---|---|---|
| `"out"` (기본) | 내가 **가리키는** 쪽 | **0건** — 프로젝트가 가리키는 대상은 없다 |
| `"in"` | 나를 **가리키는** 쪽 | `킥오프 -[part_of]-> 프로젝트` 등 5건 |
| `"both"` | 양쪽 (Cypher 의 무방향 `-[]-`) | 5건 |

문서의 태그(`프로젝트`·`RAG` …)는 `문서 -[mentions]-> 태그` 처럼 화살표를 **받기만** 하는 노드라
`direction="both"`(또는 `"in"`) 로 봐야 이웃이 보입니다. 방향을 잘못 잡으면 그래프가 텅 빈 것처럼 보입니다.

**② 홉(hop)** — 엣지를 **한 번 건너는 것**이 1홉입니다.

```
권한 ──part_of──> 보안 ──constrains──> 프로젝트
 └──── 1홉 ────┘
 └──────────────── 2홉 ────────────────┘
```

- **1홉** = 직접 이웃 (`query_neighbors`)
- **2홉** = 이웃의 이웃 — 어느 문서에도 적혀 있지 않은 `권한 → 프로젝트` 연결이 여기서 나온다
- `max_hops` 를 키우면 더 멀리 닿지만 경로 수가 급격히 늘고 관련성은 떨어진다(실무는 보통 **2~3홉**)

`multi_hop_query()` 는 가까운 홉부터 넓혀 가므로(BFS) 각 노드는 **가장 짧은 경로로 한 번만** 등장합니다.
같은 탐색의 Cypher 판이 2장의 `-[:PART_OF|CONSTRAINS|…*1..2]-` 이며, `*1..2` 가 곧 "1~2홉"입니다.

**여기서 이미 Vector RAG 가 못 하는 일이 보입니다.**

`권한 → 보안 → 프로젝트` 경로는 어느 문서에도 그렇게 적혀 있지 않습니다.
관계를 따라간 **추론** 의 결과입니다. 벡터 유사도로는 이런 연결을 만들 수 없습니다.

다만 대가가 있습니다 — **이 관계들은 누군가 설계해야 합니다.**
위 `TOPIC_LINKS` 는 사람이 손으로 적었습니다. 4장에서는 이 작업을 LLM 에게 맡깁니다.

---
## 2. Neo4j 벡터 인덱스 — 하나의 DB 에서 벡터 + 그래프

Neo4j 는 그래프 DB 로 알려져 있지만 **5.11 부터 네이티브 벡터 인덱스** 를 지원합니다.
즉 임베딩과 관계를 한 DB 에 함께 둘 수 있습니다.

```
(:Document {text, embedding}) ──[:MENTIONS]──> (:Topic) ──[관계]──> (:Topic)
        ▲                                            │
   벡터 인덱스로 시맨틱 검색                    그래프로 근거 확장
```

이것이 Graph RAG 의 가장 실용적인 형태입니다.
① 벡터로 "질문과 비슷한 문서"를 찾고 → ② 관계를 타고 "연결된 다른 근거"까지 끌어옵니다.

In [4]:
# Neo4j 컨테이너 기동 — 이미 떠 있으면 재사용한다 (CMD 문법)
import time

NEO4J_URI      = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")  # 화면에 출력하지 않는다

rc, out = utils.run_cmd('docker ps --filter name=neo4j --format "{{.Names}} {{.Status}}"',
                        echo=False)
if "neo4j" in (out or ""):
    print(f"이미 실행 중: {out.strip()}")
else:
    utils.run_cmd("docker rm -f neo4j", echo=False)  # 중지된 동명 컨테이너 정리
    # APOC 플러그인 필수(LlamaIndex 의 Neo4jPropertyGraphStore 가 apoc.meta.data 를 호출)
    utils.run_cmd('docker run -d --name neo4j -p 7474:7474 -p 7687:7687 '
                  '-e NEO4J_AUTH=neo4j/password '
                  '-e "NEO4J_PLUGINS=[\\"apoc\\"]" neo4j:5-community')
    print("컨테이너 기동 — Bolt 포트가 열릴 때까지 대기(최대 90초)")
    for _ in range(30):
        rc, _out = utils.run_cmd('curl.exe -s -o NUL -w "%{http_code}" '
                                 'http://localhost:7474', echo=False)
        if rc == 0:
            break
        time.sleep(3)

print(f"\nNeo4j URI: {NEO4J_URI}  (브라우저 콘솔: http://localhost:7474)")

이미 실행 중: neo4j Up 25 hours

Neo4j URI: bolt://localhost:7687  (브라우저 콘솔: http://localhost:7474)


In [5]:
# Neo4j 를 벡터 DB 로: 연결 → 벡터 인덱스 생성 → 문서+임베딩+태그 적재
store = rag.Neo4jVectorStore(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
NEO4J_READY = store.connected

if NEO4J_READY:
    store.clear()               # 반복 실행을 멱등하게
    store.drop_vector_index()   # 임베더가 바뀌어도 차원 충돌이 없도록 재생성
    store.setup_schema(dim=embedder.dim, similarity="cosine")

    # 문서 본문을 통째로 임베딩(문서 단위 그래프를 만들기 위해)
    texts = [doc.body for doc in docs]
    topics = [doc.tags for doc in docs]
    store.add_documents(texts, embedder.encode(texts, kind="passage"), topics=topics)
    store.add_topic_links(TOPIC_LINKS)

    print(f"\n적재 요약: {store.stats()}")
else:
    print("Neo4j 미연결 — 2장과 4장의 Neo4j 부분은 건너뜁니다")

Neo4j 연결 성공: bolt://localhost:7687
기존 Document/Topic 노드 삭제 완료


스키마 준비 완료 — 벡터 인덱스 'doc_embeddings' (dim=384, cosine)


문서 6개 적재 완료 (주제 태그 포함)
주제 관계 6개 생성 완료

적재 요약: {'docs': 6, 'topics': 15, 'rels': 461}


### 2-1. Cypher 로 시맨틱 검색

벡터 인덱스 검색은 Cypher 프로시저 한 줄입니다.

```cypher
CALL db.index.vector.queryNodes('doc_embeddings', 3, $queryVector)
YIELD node, score
RETURN node.text AS text, score ORDER BY score DESC
```

`$queryVector` 는 **인덱스를 만든 것과 같은 모델로** 임베딩한 질문 벡터여야 합니다.

In [6]:
# 벡터 인덱스 시맨틱 검색
if NEO4J_READY:
    question = "문서 등급과 접근 권한은 어떻게 정하나?"
    query_vec = embedder.encode_one(question, kind="query")

    print(f"질문: {question}\n")
    print("=== Neo4j 벡터 검색 Top-3 ===")
    for hit in store.similarity_search(query_vec, k=3):
        print(f"  [{hit['score']:.3f}] {hit['doc_id']}  {hit['text'][:60]}…")

    print("\n=== 같은 검색을 원시 Cypher 로 ===")
    for row in store.run("""
        CALL db.index.vector.queryNodes('doc_embeddings', 2, $vector)
        YIELD node, score
        RETURN node.doc_id AS doc_id, round(score, 3) AS score
        """, vector=query_vec.tolist()):
        print(f"  {row}")

질문: 문서 등급과 접근 권한은 어떻게 정하나?

=== Neo4j 벡터 검색 Top-3 ===
  [0.777] doc-2  # 사내 문서 보안 지침

## 등급 구분
문서는 공개, 사내한정, 대외비 세 등급으로 나눈다.
등급은 문서…
  [0.740] doc-0  # 사내 문서 검색 시스템 킥오프 회의록

## 참석자
개발팀 4명, 기획팀 2명이 참석했다. 외부 자문으로…
  [0.651] doc-3  # 2026년 12주차 주간 업무 보고

## 이번 주 한 일
문서 수집 파이프라인의 첫 버전을 만들었다. …

=== 같은 검색을 원시 Cypher 로 ===
  {'doc_id': 'doc-2', 'score': 0.777}
  {'doc_id': 'doc-0', 'score': 0.74}


### 2-2. 벡터 + 그래프 확장 = Graph RAG

`graph_expanded_search()` 는 두 단계로 동작합니다.

1. **씨앗(seed)**: 벡터 인덱스로 가장 가까운 문서 k개를 찾는다
2. **확장(expand)**: 씨앗 문서의 태그 → 관계를 타고 N홉까지 넓힌 태그 →
   그 태그를 가진 *다른* 문서를 근거로 추가한다

벡터만으로는 순위 밖이던 문서가 관계 덕분에 올라옵니다.

In [7]:
# Graph RAG: 벡터로 씨앗 → 태그 그래프로 근거 확장
if NEO4J_READY:
    question = "프로젝트 진행 상황과 관련된 기록"
    query_vec = embedder.encode_one(question, kind="query")
    result = store.graph_expanded_search(query_vec, k=2, hops=2)

    print(f"질문: {question}\n")
    print("① 벡터 검색으로 찾은 씨앗 문서")
    for seed in result["seeds"]:
        print(f"   [{seed['score']:.3f}] {seed['doc_id']}")

    print(f"\n② 씨앗의 태그 + 2홉 이내 관련 태그")
    print(f"   {result['topics']}")

    print("\n③ 그래프 관계로 추가 확보한 문서 (벡터 상위권 밖이던 문서)")
    for extra in result["expanded"][:4]:
        print(f"   + {extra['doc_id']} (공유 태그 {extra['overlap']}개 {extra['via_topics']})")

    print("\n→ 벡터 검색만 했다면 놓쳤을 문서를 '관계'가 데려온다")

질문: 프로젝트 진행 상황과 관련된 기록

① 벡터 검색으로 찾은 씨앗 문서
   [0.720] doc-2
   [0.675] doc-0

② 씨앗의 태그 + 2홉 이내 관련 태그
   ['LCEL', 'LangChain', 'RAG', '권한', '규정', '보안', '주간보고', '킥오프', '프로젝트']

③ 그래프 관계로 추가 확보한 문서 (벡터 상위권 밖이던 문서)
   + doc-3 (공유 태그 3개 ['RAG', '주간보고', '프로젝트'])
   + doc-5 (공유 태그 3개 ['RAG', 'LCEL', 'LangChain'])

→ 벡터 검색만 했다면 놓쳤을 문서를 '관계'가 데려온다


In [8]:
# 순수 Cypher 실습 — 집계 / 경로 / 공통 이웃
if NEO4J_READY:
    print("=== ① 태그별 문서 수 (집계) ===")
    for row in store.run("""
        MATCH (t:Topic)<-[:MENTIONS]-(d:Document)
        RETURN t.name AS topic, count(d) AS docs
        ORDER BY docs DESC, topic LIMIT 5
        """):
        print(f"   {row['topic']:<12} {row['docs']}개 문서")

    print("\n=== ② '권한' 에서 출발한 2홉 경로 (multi-hop) ===")
    for row in store.run("""
        MATCH path = (a:Topic {name: $start})-[:PART_OF|CONSTRAINS|DEFINES|NEEDS|REPORTS*1..2]-(b:Topic)
        WHERE a <> b
        RETURN [n IN nodes(path) | n.name] AS hop_path, length(path) AS hops
        ORDER BY hops LIMIT 5
        """, start="권한"):
        print(f"   {row['hops']}홉: " + " → ".join(row["hop_path"]))

    print("\n=== ③ 같은 태그를 공유하는 문서 쌍 (공통 이웃) ===")
    for row in store.run("""
        MATCH (d1:Document)-[:MENTIONS]->(t:Topic)<-[:MENTIONS]-(d2:Document)
        WHERE d1.doc_id < d2.doc_id
        RETURN d1.doc_id AS a, d2.doc_id AS b, collect(t.name) AS shared
        ORDER BY size(shared) DESC LIMIT 5
        """):
        print(f"   {row['a']}\n   ↔ {row['b']}   공유: {row['shared']}\n")

=== ① 태그별 문서 수 (집계) ===
   RAG          3개 문서
   보안           2개 문서
   프로젝트         2개 문서
   LCEL         1개 문서
   LangChain    1개 문서

=== ② '권한' 에서 출발한 2홉 경로 (multi-hop) ===
   1홉: 권한 → 보안
   2홉: 권한 → 보안 → 프로젝트
   2홉: 권한 → 보안 → 규정

=== ③ 같은 태그를 공유하는 문서 쌍 (공통 이웃) ===
   doc-0
   ↔ doc-3   공유: ['RAG', '프로젝트']

   doc-0
   ↔ doc-2   공유: ['보안']

   doc-0
   ↔ doc-5   공유: ['RAG']

   doc-3
   ↔ doc-5   공유: ['RAG']



---
## 3. LlamaIndex 기반 RAG

LangChain 과 함께 RAG 의 양대 프레임워크가 **LlamaIndex** 입니다.
2편에서 LangChain 으로 손수 조립했던 것(청킹 → 임베딩 → 저장 → 검색 → 프롬프트)을
LlamaIndex 는 `VectorStoreIndex.from_documents()` **한 줄** 로 처리합니다.

| | LangChain | LlamaIndex |
|---|---|---|
| 철학 | 범용 **오케스트레이션**(부품을 내가 조립) | **인덱싱/검색 특화**(관례를 따르면 알아서) |
| 추상화 | 낮음 — 단계가 다 보임 | 높음 — 한 줄로 끝, 대신 내부가 덜 보임 |
| 강점 | 유연한 파이프라인, 에이전트 | 인덱스 종류(벡터/그래프/요약)가 풍부 |

### 문제: LlamaIndex 는 자체 LLM/임베딩 추상화를 갖고 있다

그대로 쓰면 `OpenAI()` 같은 특정 공급자를 코드에 박게 되어 이 강의의 `.env` 규약이 깨집니다.
그래서 [`agentic_lib/llama_rag.py`](agentic_lib/llama_rag.py) 에 **어댑터 두 개** 를 만들어 두었습니다.

```
utils.get_llm()            →  LangChainLLM      (LlamaIndex CustomLLM)
embeddings.get_embedder()  →  InjectedEmbedding (LlamaIndex BaseEmbedding)
```

덕분에 `llama-index-llms-*` 패키지를 하나도 설치하지 않고 ollama·google·nvidia·openrouter
어디서나 동일하게 동작합니다.

In [9]:
# LlamaIndex 전역 설정에 이 강의의 LLM/임베딩을 꽂는다
llama_rag.configure(llm, embedder, chunk_size=400, chunk_overlap=80)

# PersonalDoc → LlamaIndex Document (메타데이터를 그대로 실어 보낸다)
documents = llama_rag.to_documents(docs)
print(f"\nDocument {len(documents)}개")
print(f"  첫 문서 메타데이터: {documents[0].metadata}")

LlamaIndex 설정 완료 — LLM: ChatOpenAI / 임베딩: local:paraphrase-multilingual-MiniLM-L12-v2 (384차원) / 청크: 400

Document 6개
  첫 문서 메타데이터: {'title': '사내 문서 검색 시스템 킥오프 회의록', 'category': '회의록', 'created': '2026-03-14', 'author': '신성태', 'tags': '프로젝트,킥오프,RAG,보안', 'source': 'workspace\\personal_docs\\2026-03-14_회의록_프로젝트킥오프.md'}


In [10]:
# VectorStoreIndex — 청킹·임베딩·인덱싱이 이 한 줄에 다 들어 있다
import time

started = time.perf_counter()
vector_index = llama_rag.build_vector_index(documents)
print(f"VectorStoreIndex 구축 {time.perf_counter() - started:.1f}s\n")

llama_rag.ask(vector_index, "킥오프 회의에서 정한 프로토타입 기간은?", top_k=3)
llama_rag.ask(vector_index, "무릎이 아팠을 때 어떻게 대처했나?", top_k=3)

VectorStoreIndex 구축 0.1s



질문: 킥오프 회의에서 정한 프로토타입 기간은?

근거 노드 3건:
  [1] (0.551) 사내 문서 검색 시스템 킥오프 회의록: 2. 임베딩은 사내망에서 동작해야 하므로 오프라인 모델을 우선 검토한다. 3. 문서 권한이 부서별로 다르므로 메타데이터에 부서…
  [2] (0.476) 사내 문서 검색 시스템 킥오프 회의록: # 사내 문서 검색 시스템 킥오프 회의록 ## 참석자 개발팀 4명, 기획팀 2명이 참석했다. 외부 자문으로 데이터 플랫폼팀 1…
  [3] (0.424) 2026년 12주차 주간 업무 보고: # 2026년 12주차 주간 업무 보고 ## 이번 주 한 일 문서 수집 파이프라인의 첫 버전을 만들었다. 마크다운과 텍스트 파…

답변:
6주
------------------------------------------------------------------------------------


질문: 무릎이 아팠을 때 어떻게 대처했나?

근거 노드 3건:
  [1] (0.243) 3월 러닝 기록: # 3월 러닝 기록 ## 총량 한 달 동안 열두 번 달렸고 누적 거리는 96킬로미터였다. 가장 길게 달린 날은 15킬로미터였고…
  [2] (0.132) 오사카 여행 준비 메모: # 오사카 여행 준비 메모 ## 일정 3박 4일이며 첫날은 저녁 도착이라 숙소 근처만 둘러보기로 했다. 둘째 날은 교토 당일치…
  [3] (0.119) 2026년 12주차 주간 업무 보고: # 2026년 12주차 주간 업무 보고 ## 이번 주 한 일 문서 수집 파이프라인의 첫 버전을 만들었다. 마크다운과 텍스트 파…

답변:
무릎이 시큰거렸을 때는 사흘간 휴식을 취하고, 그 후에 신발을 바꾸어 착용한 뒤 통증이 완화되었다.
------------------------------------------------------------------------------------


'무릎이 시큰거렸을 때는 사흘간 휴식을 취하고, 그 후에 신발을 바꾸어 착용한 뒤 통증이 완화되었다.'

---
## 4. GraphRAG — LLM 이 지식 그래프를 자동 추출

1장에서는 관계(`TOPIC_LINKS`)를 사람이 손으로 적었습니다.
**PropertyGraphIndex** 는 그 작업을 LLM 에게 맡깁니다.

```
문서 청크 → [LLM: "이 텍스트에서 (주어, 관계, 목적어)를 뽑아라"] → 트리플 → 그래프 저장
```

이것이 Microsoft 의 GraphRAG 논문이 대중화한 방식이고, LlamaIndex 의 구현이
`SimpleLLMPathExtractor` 입니다.

이 장은 두 단계를 이어서 봅니다 — **① 트리플 자동 추출**(4장 본문) →
**② 커뮤니티 요약으로 전역 질문에 답하기**(4-1절, Microsoft GraphRAG 의 Global Search).

> ⏱️ **시간이 걸립니다.** 청크마다 LLM 을 호출하므로 문서 3개에 1~2분이 걸릴 수 있습니다.
> 그래서 아래에서는 문서를 3개로 제한합니다.
>
> ⚠️ **그래프 품질 = 추출 LLM 품질.** 작은 모델은 프롬프트의 예시를 그대로 베끼거나
> 엉뚱한 트리플을 만듭니다. 실행 결과에서 직접 확인합니다.

In [11]:
# PropertyGraphIndex — LLM 이 문서에서 트리플을 뽑아 Neo4j 에 저장한다
neo4j_config = ({"uri": NEO4J_URI, "user": NEO4J_USER, "password": NEO4J_PASSWORD}
                if NEO4J_READY else None)
if neo4j_config is None:
    print("Neo4j 미연결 — 메모리 내 그래프 저장소로 진행합니다")

started = time.perf_counter()
graph_index = llama_rag.build_property_graph_index(
    documents[:3],                 # 시간 관계상 3개만 (전체는 2배 이상 걸린다)
    neo4j=neo4j_config,
    max_triplets_per_chunk=6,
)
print(f"\nPropertyGraphIndex 구축 {time.perf_counter() - started:.1f}s")

Neo4j 그래프 저장소 연결: bolt://localhost:7687



PropertyGraphIndex 구축 93.7s


In [12]:
# LLM 이 실제로 무엇을 뽑아냈는지 눈으로 확인한다
llama_rag.print_extracted_triplets(graph_index, limit=20)

LLM 이 자동 추출한 관계 240개 (상위 20개)
  (사내 문서 검색 시스템 킥오프 회의록) -[Title]-> (사내 문서 검색 시스템 킥오프 회의록)
  (사내 문서 검색 시스템 킥오프 회의록) -[Source]-> (Workspace\personal_docs\2026-03-14_회의록_프로젝트킥오프.md)
  (사내 문서 검색 시스템 킥오프 회의록) -[Is]-> (회의록)
  (사내 문서 검색 시스템 킥오프 회의록) -[Author]-> (신성태)
  (사내 문서 검색 시스템 킥오프 회의록) -[Has tag]-> (프로젝트)
  (사내 문서 검색 시스템 킥오프 회의록) -[Created on]-> (2026-03-14)
  (사내 문서 검색 시스템 킥오프 회의록) -[Created]-> (2026-03-14)
  (사내 문서 검색 시스템 킥오프 회의록) -[Category]-> (회의록)
  (개발팀) -[Attended]-> (사내 문서 검색 시스템 킥오프 회의록)
  (신성태) -[Authored]-> (사내 문서 검색 시스템 킥오프 회의록)
  (Title) -[Is]-> (사내 문서 검색 시스템 킥오프 회의록)
  (신성태) -[Is author of]-> (사내 문서 검색 시스템 킥오프 회의록)
  (신성태) -[Author of]-> (사내 문서 검색 시스템 킥오프 회의록)
  (신성태) -[Author]-> (사내 문서 검색 시스템 킥오프 회의록)
  (신성태) -[Author_of]-> (사내 문서 검색 시스템 킥오프 회의록)
  (회의록) -[Created on]-> (2026-03-14)
  (개발팀) -[Attended]-> (회의록)
  (신성태) -[Authored]-> (회의록)
  (Created) -[Is]-> (2026-03-14)
  (사내 문서 검색 시스템) -[Created on]-> (2026-03-14)

※ 프롬프트 예시가 그대로 복사된 트리플 2개는 숨겼습니다(예: Subject → Object).
  

In [13]:
# GraphRAG 로 질의 — 그래프를 근거로 답한다
# 근거 노드는 'LLM 이 추출한 트리플 + 원본 청크' 가 합쳐진 형태다(그래서 미리보기에 트리플이 먼저 나온다)
llama_rag.ask(graph_index, "킥오프 회의에서 무엇을 결정했나?", top_k=3)

질문: 킥오프 회의에서 무엇을 결정했나?

근거 노드 2건:
  [1] (0.690) 사내 문서 검색 시스템 킥오프 회의록: 개발팀 -> Attended -> Meeting 개발팀 -> Attended -> 회의 # 사내 문서 검색 시스템 킥오프 회의…
  [2] (0.678) 사내 문서 검색 시스템 킥오프 회의록: 사내 문서 검색 시스템 -> Is -> 킥오프 회의록 사내 문서 검색 시스템 -> Is part of -> 킥오프 회의록 # …

답변:
킥오프 회의에서 다음과 같이 결정했습니다.

1. **1차 목표**: 사내 규정 문서와 회의록을 대상으로 하는 시맨틱 검색을 구현한다.  
2. **임베딩 모델**: 사내망 내에서 동작해야 하므로 오프라인(온‑프레미스) 모델을 우선 검토한다.  

(세 번째 결정 사항은 문서에 구체적으로 명시되어 있지 않습니다.)
------------------------------------------------------------------------------------


'킥오프 회의에서 다음과 같이 결정했습니다.\n\n1. **1차 목표**: 사내 규정 문서와 회의록을 대상으로 하는 시맨틱 검색을 구현한다.  \n2. **임베딩 모델**: 사내망 내에서 동작해야 하므로 오프라인(온‑프레미스) 모델을 우선 검토한다.  \n\n(세 번째 결정 사항은 문서에 구체적으로 명시되어 있지 않습니다.)'

In [14]:
# 같은 질문을 Vector RAG 와 GraphRAG 에 각각 던져 비교
llama_rag.compare_vector_and_graph(vector_index, graph_index, [
    "프로토타입 데모까지 기간은?",              # 단일 사실 — Vector 가 유리
    "회의록과 보안 지침은 어떻게 이어지나?",     # 관계 질문 — Graph 의 영역
], top_k=3)


질문: 프로토타입 데모까지 기간은?



[Vector RAG] 39.1s
  We need to answer: "프로토타입 데모까지 기간은?"<unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk



[GraphRAG] 15.5s
  제공된 문서에는 프로토타입 데모까지의 기간에 대한 정보가 포함되어 있지 않습니다. 따라서 해당 기간을 알 수 없습니다.


질문: 회의록과 보안 지침은 어떻게 이어지나?



[Vector RAG] 3.9s
  회의록에서는 사내 문서 검색 시스템을 구축하면서 **문서의 등급(공개·사내한정·대외비)과 부서별 접근 범위를 메타데이터에 반드시 포함**하고, **오프라인 임베딩 모델을 사용**하여 6주 안에 프로토타입을 만들겠다고 결정했습니다.  
이는 보안 지침에서 정한 **“검색 색인에는 등급 정보를 반드시 함께 저장하고, 사용자가 볼 수 없는 등급의 문서는 검색 결과에 나타나지 않아야 한다”**는 요구사항을 시스템 설계에 반영하려는 연결점입니다. 즉, 회의록은 보안 지침의 등급‑기반 검색 제한 원칙을 실제 검색 시스템에 구현하기 위한 구체적



[GraphRAG] 31.8s
  회의록에서는 사내 문서 검색 시스템(시맨틱 검색) 구축을 kick‑off 하면서 보안(보안 태그)을 고려 대상 중 하나로 명시하고 있습니다.  
이때 제시된 보안 지침은 바로 그 검색 시스템이 따라야 할 구체적인 보안 규칙을 정의한 문서입니다. 즉,

- 회의록: “1차 목표는 사내 규정 문서와 회의록을 대상으로 하는 시맨틱 검색이며, 임베딩은 사내망에서 동작하는 오프라인 모델을 검토한다”고 결정하고, 태그에 **보안**을 포함시킴.  
- 보안 지침: 검색 색인에 문서 등급(공개/사내한정/대외비) 정보를 반드시 저장하고, 사용자가



### 4-1. 커뮤니티 요약과 전역(Global) 질문 — Microsoft GraphRAG

방금 만든 그래프로도 답하기 어려운 질문이 있습니다. **전체를 조망하는 질문** 입니다.

> "이 문서 모음이 다루는 주요 주제가 뭐야?"

이 질문의 답은 어느 한 청크에도 없습니다. 벡터 검색은 상위 k개만 보고, 그래프 탐색도
출발 엔티티 주변만 봅니다. 원리적으로 **전부를 봐야** 답할 수 있는 질문입니다.

Microsoft 가 오픈소스로 공개한 **GraphRAG** 의 핵심 아이디어가 여기에 있습니다.

```
문서 → 트리플 추출 → 지식 그래프 → [커뮤니티 탐지] → 커뮤니티별 요약(미리 만들어 둠)
                                                          ↓
     전역 질문 ──────────────────────────────→ 요약들을 모아 답변 (Global Search)
     특정 질문 ──────────────────────────────→ 관련 엔티티 주변만 (Local Search)
```

아래에서 **앞 절에서 만든 그래프**(`graph_index`)를 재료로 이 흐름을 재현합니다.

> 원조는 커뮤니티를 **Leiden** 알고리즘으로 계층적으로 나눕니다. 여기서는 이미 설치된
> `networkx` 의 **Louvain** 으로 한 단계(비계층)만 재현합니다 — 아이디어는 같고 구현이 가볍습니다.

In [15]:
# 앞 절에서 LLM 이 자동 추출한 그래프를 '커뮤니티(주제 클러스터)'로 나눈다
triplets = gstack.triplets_from_graph_index(graph_index)
print(f"추출된 트리플 {len(triplets)}개\n")

communities = gstack.detect_communities(triplets, min_size=2)
gstack.print_communities(communities, limit=5)

추출된 트리플 242개



커뮤니티 35개 탐지 (상위 5개)

[C0] 엔티티 32개 / 관계 53개
   12, 120 km, 120킬로미터, 12번, 15 km, 2026-03-31, 3, 3월 러닝 기록 외 24개

[C1] 엔티티 20개 / 관계 22개
   Held, Internal regulation documents, Internal regulation documents and meeting minutes, Kickoff meeting, Offline embedding model, Offline model, Rag, Regulation documents 외 12개

[C2] 엔티티 17개 / 관계 29개
   2026-03-14, 4, Author, Created, Meeting, Tags, Title, Workspace\personal_docs\2026-03-14_회의록_프로젝트킥오프.md 외 9개

[C3] 엔티티 14개 / 관계 22개
   Documents, Scattered, 공개, 대외비, 등급, 등급 미지정 시, 문서, 문서 등급 외 6개

[C4] 엔티티 7개 / 관계 9개
   2, Planning team, 공유, 기획팀, 부서별 문서 접근 정책, 부서별 문서 접근 정책 정리, 부서별 문서 접근 정책 정리 및 공유



In [16]:
# 커뮤니티별 요약 생성 → 그 요약들만으로 전역 질문에 답한다
print("=== 커뮤니티 요약 (Microsoft GraphRAG 의 community report 단계) ===")
summarized = gstack.summarize_communities(llm, communities, limit=4)

GLOBAL_Q = "이 문서 모음이 다루는 주요 주제는 무엇인가?"
gstack.global_answer(llm, summarized, GLOBAL_Q)

=== 커뮤니티 요약 (Microsoft GraphRAG 의 community report 단계) ===


[C0] 3월 러닝 기록은 2026-03-31에 작성된 개인 메모로, 월간 12회·96km를 달리며 평균 페이스 5분40초/km을 기록했으며, 둘째 주에 무릎 불편감이 있었고 아침 러닝 루틴을 가벼운 식사 후로 변경하였다. 


[C1] 이 클러스터는 사내 문서 검색 시스템이 내부 규정 문서와 회의록에 대한 시맨틱 검색을 목표로 하는 내용을 담고 있다. 시스템은 평균 20분 소요되는 검색 문제를 오프라인 임베딩 모델을 활용해 해결하며, 킥오프 회의와


[C2] 이 클러스터는 '사내 문서 검색 시스템 킥오프 회의록'이라는 제목의 회의록을 다루며, 작성자는 신성태이고 작성일은 2026-03-14이다. 회의에는 개발팀(4명)이 참석하여 다음 주까지 문서 수집 파이프라인 초안을 


[C3] 사내 위키와 공유 드라이브에 저장된 문서들은 흩어져 있어 필요한 정보를 찾는 데 평균 20분이 소요되며, 문서의 등급은 작성자가 지정하고 지정되지 않을 경우 사내한정으로 간주된다. 문서 등급에는 공개, 사내한정, 대



질문(전역): 이 문서 모음이 다루는 주요 주제는 무엇인가?
답변:
이 문서 모음의 주요 주제는 **사내 문서 검색 시스템 구축 및 관련 문서 관리**입니다. 구체적으로는 내부 규정 문서와 회의록에 대한 시맨틱 검색을 목표로 하는 시스템(클러스터 1), 해당 시스템의 킥오프 회의록 및 업무 배분(클러스터 2), 그리고 사내 위키·공유 드라이브에 흩어진 문서들의 등급(공개, 사내한정, 대외비) 및 검색 소요 시간 문제(클러스터 3)가 중심 내용입니다. (클러스터 0의 개인 러닝 기록은 별도의 메모로, 주요 주제와 직접적인 관련은 없습니다.)


'이 문서 모음의 주요 주제는 **사내 문서 검색 시스템 구축 및 관련 문서 관리**입니다. 구체적으로는 내부 규정 문서와 회의록에 대한 시맨틱 검색을 목표로 하는 시스템(클러스터\u202f1), 해당 시스템의 킥오프 회의록 및 업무 배분(클러스터\u202f2), 그리고 사내 위키·공유 드라이브에 흩어진 문서들의 등급(공개, 사내한정, 대외비) 및 검색 소요 시간 문제(클러스터\u202f3)가 중심 내용입니다. (클러스터\u202f0의 개인 러닝 기록은 별도의 메모로, 주요 주제와 직접적인 관련은 없습니다.)'

In [17]:
# 같은 전역 질문을 Vector RAG 에 던져 비교한다 — 상위 k개 청크만 보는 한계
print("=== 같은 질문, Vector RAG (3장) ===")
llama_rag.ask(vector_index, GLOBAL_Q, top_k=3)

=== 같은 질문, Vector RAG (3장) ===


질문: 이 문서 모음이 다루는 주요 주제는 무엇인가?

근거 노드 3건:
  [1] (0.606) 사내 문서 검색 시스템 킥오프 회의록: # 사내 문서 검색 시스템 킥오프 회의록 ## 참석자 개발팀 4명, 기획팀 2명이 참석했다. 외부 자문으로 데이터 플랫폼팀 1…
  [2] (0.593) 사내 문서 검색 시스템 킥오프 회의록: 2. 임베딩은 사내망에서 동작해야 하므로 오프라인 모델을 우선 검토한다. 3. 문서 권한이 부서별로 다르므로 메타데이터에 부서…
  [3] (0.574) 2026년 12주차 주간 업무 보고: # 2026년 12주차 주간 업무 보고 ## 이번 주 한 일 문서 수집 파이프라인의 첫 버전을 만들었다. 마크다운과 텍스트 파…

답변:
이 문서 모음은 사내 문서 검색 시스템을 구축하는 프로젝트와 그 진행 상황을 다루고 있습니다. 구체적으로는 킥오프 회의에서 정한 목표(시맨틱 검색, 오프라인 임베딩 모델 검토, 메타데이터에 부서·공개 범위 포함, 6주 내 프로토타입 제작)와 주간 업무 보고에서 이루어진 문서 수집 파이프라인 구축, 임베딩 모델 비교, 청크 크기 조정, 벡터 DB 검토 및 권한 필터 검증 등의 활동이 주요 내용입니다. 따라서 전체적인 주제는 **사내 문서 검색 시스템(RAG 기반 시맨틱 검색) 개발 프로젝트**입니다.
------------------------------------------------------------------------------------


'이 문서 모음은 사내 문서 검색 시스템을 구축하는 프로젝트와 그 진행 상황을 다루고 있습니다. 구체적으로는 킥오프 회의에서 정한 목표(시맨틱 검색, 오프라인 임베딩 모델 검토, 메타데이터에 부서·공개 범위 포함, 6주 내 프로토타입 제작)와 주간 업무 보고에서 이루어진 문서 수집 파이프라인 구축, 임베딩 모델 비교, 청크 크기 조정, 벡터 DB 검토 및 권한 필터 검증 등의 활동이 주요 내용입니다. 따라서 전체적인 주제는 **사내 문서 검색 시스템(RAG 기반 시맨틱 검색) 개발 프로젝트**입니다.'

### 4-2. 실행 결과를 정직하게 읽기

**① 이 데이터셋에서는 Vector RAG 도 꽤 잘 답합니다.**
문서가 **6개뿐** 이라 상위 3개 청크만 봐도 코퍼스의 절반을 본 셈이기 때문입니다.
커뮤니티 요약의 값어치는 **문서가 수백~수천 건일 때** 드러납니다 — 그때는 top-k 가
전체의 1%도 못 덮으므로, 미리 만들어 둔 요약 없이는 전역 질문에 답할 방법이 없습니다.
지금 단계에서 확인할 것은 성능 우열이 아니라 **파이프라인이 다르다** 는 사실입니다.

**② 커뮤니티 품질 = 트리플 추출 품질.**
`12`, `120 km`, `4` 같은 숫자가 엔티티로 잡혀 클러스터에 섞여 있다면, 그것은 커뮤니티
탐지의 문제가 아니라 **트리플 추출 LLM 의 문제** 입니다. 작은 모델일수록 값(value)과
엔티티(entity)를 구분하지 못합니다. 이 장 첫머리의 경고가 여기까지 그대로 이어집니다.

### 4-3. 진짜 Microsoft GraphRAG 를 쓸 때

위 실습은 **아이디어를 재현** 한 것입니다. 실제 패키지는 훨씬 많은 일을 합니다.

```bat
REM 공식 구현 (인덱싱 파이프라인 + Local/Global Search)
uv pip install graphrag

graphrag init --root .\ragtest
REM  → settings.yaml 에 LLM/임베딩 설정을 적고 input\ 에 문서를 넣는다
graphrag index --root .\ragtest
graphrag query --root .\ragtest --method global --query "주요 주제가 뭐야?"
graphrag query --root .\ragtest --method local  --query "무릎 통증은 어떻게 해결했나?"
```

| 우리 재현 | 공식 graphrag |
|---|---|
| Louvain 1단계 | **Leiden 계층형** — 커뮤니티 안에 다시 커뮤니티 |
| 커뮤니티 요약 1회 | 계층별 요약 + **중요도 점수** |
| 요약을 한 번에 넣고 답변 | **map-reduce** — 요약별 부분 답변 → 종합 |
| 우리 `.env` 공급자 그대로 | `settings.yaml` 로 별도 설정(OpenAI 중심) |

**비용을 먼저 계산하세요.** 인덱싱에 문서당 LLM 을 수십 번 호출합니다.
문서 수백 건이면 시간과 토큰이 상당히 듭니다.

**언제 값을 하나**
- "전체 경향", "주요 쟁점", "이 조직에서 반복 등장하는 주제" 류의 **전역 질문** 이 잦을 때
- 답변에 **근거 경로** 를 제시해야 할 때
- 반대로 단일 사실 조회가 대부분이면 → 2편의 Vector RAG 로 충분합니다

### 4-4. GraphRAG 를 언제 쓸 것인가

실행 결과를 보면 냉정하게 판단해야 합니다.

**GraphRAG 의 비용**
- 색인 시간이 Vector RAG 의 **수십 배**(청크마다 LLM 호출)
- 품질이 추출 LLM 에 좌우됨 — 작은 모델은 쓸모없는 트리플을 대량 생산
- 같은 개념이 다른 이름으로 여러 노드가 되는 **엔티티 중복** 문제

**그럼에도 값을 하는 경우**
- 문서 수백~수천 건에 걸쳐 **같은 엔티티가 반복 등장** 할 때(인물·조직·제품)
- "A 와 B 의 관계는?", "X 에서 시작해 Y 까지 어떻게 이어지나" 류의 질문이 잦을 때
- 답변에 **추론 경로를 근거로 제시** 해야 할 때

**실무 권장 순서**

1. Vector RAG 로 시작한다(2편).
2. 실패하는 질문을 모아 유형을 본다.
3. 실패가 대부분 "관계 질문"이면 그때 그래프를 도입한다.
4. 도입하더라도 **Hybrid** 로 — 벡터를 버리지 않는다(5장).

---
## 5. Hybrid RAG — 벡터 + 그래프

실무의 정답은 대개 둘 중 하나가 아니라 **둘 다** 입니다.

```
질문 ─┬─→ [벡터 검색]  → 비슷한 청크 k개 ──┐
      │                                    ├─→ [통합 컨텍스트] → LLM → 답변
      └─→ [엔티티 추출 → 그래프 탐색] ──────┘
```

`rag.HybridRAG` 가 이 구조를 구현합니다.

In [18]:
# Hybrid RAG: ChromaDB 벡터 검색 + KnowledgeGraph 컨텍스트
import chromadb

# 2편에서 만든 청크를 그대로 재사용
chunks = doc_prep.chunk_all(docs, strategy="header_merged")
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("hybrid_docs")
except Exception:
    pass
collection = chroma_client.create_collection(
    name="hybrid_docs", metadata={"hnsw:space": "cosine"}, embedding_function=None)
collection.add(
    ids=[c.chunk_id for c in chunks],
    documents=[c.text for c in chunks],
    metadatas=[dict(c.metadata) for c in chunks],
    embeddings=embedder.encode([c.text for c in chunks], kind="passage").tolist(),
)
print(f"ChromaDB 적재 {collection.count()}건")

# VectorRAG 는 질의 임베딩이 필요하므로 같은 임베더를 쓰는 래퍼를 끼운다
class EmbeddedCollection:
    """ChromaDB 컬렉션에 우리 임베더를 물려 주는 얇은 래퍼."""

    def __init__(self, coll, embedder):
        self._coll, self._embedder = coll, embedder

    def query(self, query_texts, n_results=3, **kwargs):
        """텍스트 질의를 우리 임베더로 벡터화해 검색한다."""
        vectors = self._embedder.encode(list(query_texts), kind="query").tolist()
        return self._coll.query(query_embeddings=vectors, n_results=n_results, **kwargs)

vector_rag = rag.VectorRAG(EmbeddedCollection(collection, embedder), llm=llm)
print("VectorRAG 준비 완료")

ChromaDB 적재 13건
VectorRAG 준비 완료


In [19]:
# Hybrid RAG — 벡터 근거 + 그래프 근거를 합쳐 답변
hybrid = rag.HybridRAG(vector_rag, kg, llm=llm)

for question in ["보안 관련 결정 사항과 규정은 어떻게 연결되나?",
                 "프로젝트와 관련된 문서들은 무엇인가?"]:
    answer = hybrid.generate_answer(question)
    print(f"\n답변: {answer[:400]}\n" + "-" * 84)

=== Hybrid RAG 검색 ===
쿼리: 보안 관련 결정 사항과 규정은 어떻게 연결되나?
발견된 엔티티: ['보안', '규정']



답변: 보안 관련 결정 사항(문서 등급 지정, 사내·외 접근 정책, 감사 기록·외부 전송 시 보고 의무 등)은 **규정_문서보안_지침**에 명시된 보안 등급 분류·저장 규칙에 따라 구체화됩니다.  

지식 그래프에서는 이 관계가 다음과 같이 표현됩니다.  

- **규정 -[defines]-> 보안**: 규정(문서보안 지침)이 보안 요구사항을 정의한다.  
- **보안 -[constrains]-> 프로젝트**: 보안 결정 사항은 프로젝트를 제약하는 방향으로 작용한다.  
- **권한 -[part_of]-> 보안**: 접근 권한 등 세부 결정 사항은 보안의 일부로 포함된다.  
- **규정_문서보안_지침 -[mentions]-> 규정** 및 **규정_문서보안_지침 -[mentions]-> 보안**: 구체적인 지침 문
------------------------------------------------------------------------------------
=== Hybrid RAG 검색 ===
쿼리: 프로젝트와 관련된 문서들은 무엇인가?
발견된 엔티티: ['프로젝트']



답변: 프로젝트와 직접 관련된 문서들은 다음과 같습니다.

**벡터 검색 결과에서 확인할 수 있는 문서**
- **사내 문서 검색 시스템 킥오프 회의록**  
  - 개발팀이 다음 주까지 문서 수집 파이프라인 초안을 만든다는 내용과, 기획팀이 부서별 문서 접근 정책을 정리해 공유한다는 내용이 포함되어 있습니다.  
  - 또한 1차 목표(시맨틱 검색 대상: 사내 규정 문서·회의록), 임베딩 모델 검토 방향, 메타데이터 요구사항, 6주 내 프로토타입 데모 계획 등이 구체적으로 기술되어 있어 프로젝트 kick‑off와 관련된 핵심 자료입니다.  
- **사내 문서 보안 지침**  
  - 대외비 문서를 외부로 전송하면 즉시 보고 대상이며 감사 기록이 남는다는 보안 규칙이 명시되어 있어, 프로젝트의 보안 제약 조건과 직
------------------------------------------------------------------------------------


---
## 6. 그래프 DB 갈아타기 — Memgraph

2장부터 여기까지 **Neo4j 하나** 만 썼습니다. 실무에서는 요구사항이 DB 를 고릅니다 —
그래프가 초 단위로 갱신되면 인메모리 엔진이, 노드가 수억 개면 분산 엔진이 어울립니다.

여기서 확인할 것은 하나입니다.

> **Memgraph 는 Neo4j 와 같은 Bolt 프로토콜 + 같은 Cypher** 를 씁니다.
> 파이썬 드라이버(`neo4j`)도, 쿼리 문자열도 그대로 두고 **접속 주소만 바꾸면** 됩니다.

아래에서 같은 코드로 두 DB 를 모두 다뤄 그 말이 사실인지 직접 확인합니다.
구현은 [`agentic_lib/graph_stack.py`](agentic_lib/graph_stack.py) 의 `BoltGraph` 입니다.

In [20]:
# Memgraph 컨테이너 기동 — Neo4j(7687)와 겹치지 않게 7688 로 매핑한다 (CMD 문법)
MEMGRAPH_URI = os.getenv("MEMGRAPH_URI", "bolt://localhost:7688")

rc, out = utils.run_cmd('docker ps --filter name=memgraph --format "{{.Names}} {{.Status}}"',
                        echo=False)
if "memgraph" in (out or ""):
    print(f"이미 실행 중: {out.strip()}")
else:
    utils.run_cmd("docker rm -f memgraph", echo=False)  # 중지된 동명 컨테이너 정리
    utils.run_cmd("docker run -d --name memgraph -p 7688:7687 memgraph/memgraph:latest")
    print("컨테이너 기동 — Bolt 포트가 열릴 때까지 대기(최대 60초)")
    for _ in range(20):
        time.sleep(3)
        # quiet=True: 기동 중에는 연결 실패가 정상이므로 메시지를 찍지 않는다
        probe = gstack.BoltGraph(MEMGRAPH_URI, quiet=True)
        if probe.connected:
            probe.close()
            break

# Memgraph 는 기본적으로 인증이 없어 사용자/비밀번호가 빈 문자열이다
print(f"\nMemgraph URI: {MEMGRAPH_URI} (인증 없음)")

이미 실행 중: memgraph Up About an hour

Memgraph URI: bolt://localhost:7688 (인증 없음)


In [21]:
# 같은 드라이버·같은 Cypher 로 Neo4j 와 Memgraph 를 모두 다룬다
neo_graph = gstack.BoltGraph(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, label="Neo4j")
mem_graph = gstack.BoltGraph(MEMGRAPH_URI, label="Memgraph")  # 사용자/비밀번호 없음

print(f"\n버전 — Neo4j: {neo_graph.version()} / Memgraph: {mem_graph.version()}")

# 6장은 :GDoc/:GTopic 전용 라벨만 쓴다 → 2·4장이 넣어 둔 Neo4j 데이터를 건드리지 않는다
results = [gstack.benchmark_graph_db(neo_graph, docs, TOPIC_LINKS, rounds=20),
           gstack.benchmark_graph_db(mem_graph, docs, TOPIC_LINKS, rounds=20)]
print()
gstack.print_benchmark(results)

print("\n=== 똑같은 Cypher 로 2홉 탐색 ===")
for graph_db in (neo_graph, mem_graph):
    if graph_db.connected:
        print(f"[{graph_db.label}]")
        for row in graph_db.two_hop_paths("권한"):
            print(f"   {row['hops']}홉: " + " → ".join(row["names"]))

Neo4j 연결 성공: bolt://localhost:7687
Memgraph 연결 성공: bolt://localhost:7688

버전 — Neo4j: 5.26.28 / Memgraph: 3.12.0



DB                적재(ms)       2홉 탐색(ms)      노드      관계
--------------------------------------------------------
Neo4j              123.3            5.87      21      25
Memgraph            17.5            3.30      21      25

※ 데이터가 작아 절대 비교가 아니다 — 워밍업·캐시에 따라 흔들린다.

=== 똑같은 Cypher 로 2홉 탐색 ===
[Neo4j]
   1홉: 권한 → 보안
   2홉: 권한 → 보안 → 규정
   2홉: 권한 → 보안 → 프로젝트
[Memgraph]
   1홉: 권한 → 보안
   2홉: 권한 → 보안 → 규정
   2홉: 권한 → 보안 → 프로젝트


### 6-1. 읽는 법과, 다른 DB 를 띄우지 않는 이유

위 숫자는 **절대 성능 비교가 아닙니다.** 문서 6개 규모이고 컨테이너 상태·캐시·워밍업에 따라
흔들립니다. 확인할 것은 두 가지뿐입니다.

1. **같은 코드가 두 DB 에서 그대로 돌았다** — 드라이버도 Cypher 도 바꾸지 않았습니다.
2. 인메모리 엔진 쪽 **적재 지연이 눈에 띄게 짧다** — 그래프를 자주 다시 쓰는 워크로드
   (대화마다 메모리 그래프를 갱신하는 에이전트 등)에서 이 차이가 누적됩니다.

**NebulaGraph·ArangoDB 는 왜 실행하지 않나**

- NebulaGraph 는 `metad`·`storaged`·`graphd` **3개 서비스** 를 띄우고 스페이스·스키마를
  먼저 정의해야 합니다. 실습 한 절에 넣기에는 준비 비용이 너무 큽니다.
- ArangoDB 는 컨테이너 하나면 되지만 **쿼리 언어가 AQL** 이라, 지금까지 쓴 Cypher 를
  전부 다시 써야 해서 이 노트북의 주제(그래프 RAG)에서 초점이 흐려집니다.

대신 **어떤 코드가 되는지** 만 아래에서 확인하고 넘어갑니다.

In [22]:
# 실습에서 띄우지 않는 DB 들은 '어떤 코드가 되는지'만 확인한다
gstack.print_other_db_snippets()

===== NebulaGraph — 단일 노드로 감당이 안 되는 초대규모 지식 그래프 =====
# uv pip install nebula3-python
from nebula3.gclient.net import ConnectionPool
from nebula3.Config import Config

pool = ConnectionPool()
pool.init([("127.0.0.1", 9669)], Config())      # graphd 주소
session = pool.get_session("root", "nebula")
session.execute("USE personal_docs")
# nGQL — Cypher 가 아니다(GO / FETCH / LOOKUP 문법)
print(session.execute("GO 2 STEPS FROM '권한' OVER RELATES YIELD dst(edge)"))

===== ArangoDB — 문서 기반 RAG 와 GraphRAG 를 한 저장소로 합치고 싶을 때 =====
# uv pip install python-arango
from arango import ArangoClient

db = ArangoClient(hosts="http://localhost:8529").db("_system", "root", "password")
graph = db.create_graph("docs")                  # 문서 컬렉션과 같은 DB 안에 그래프를 둔다
# AQL — 문서 조회와 그래프 순회를 한 쿼리로 섞을 수 있는 것이 이 DB 의 강점
db.aql.execute('''
    FOR t IN 1..2 OUTBOUND 'topics/권한' GRAPH 'docs'
        RETURN t.name
''')



---
## 7. 자연어 → Cypher: LangChain `GraphCypherQAChain`

지금까지의 검색은 모두 **우리가 쿼리를 미리 짜 둔 것** 이었습니다.
`GraphCypherQAChain` 은 그 일을 LLM 에게 맡깁니다.

```
질문(자연어) → [LLM: 스키마를 보고 Cypher 작성] → 그래프 실행 → 결과 → [LLM: 답변 작성]
```

**왜 필요한가** — 벡터 검색은 "몇 개냐", "가장 많은 것 3개" 같은 **집계** 에 원리적으로 약합니다.
청크를 아무리 잘 찾아도 개수를 세지는 못하기 때문입니다. 그래프 DB 는 그걸 한 줄로 합니다.

**정확도를 가르는 것은 스키마 품질입니다.** 지금 Neo4j 에는 2장의 깨끗한 모델
(`Document`·`Topic`)과 4장 `PropertyGraphIndex` 가 만든 `entity`·`Chunk`·`__Node__` 가
뒤섞여 있습니다. 이대로 LLM 에게 보여 주면 엉뚱한 라벨로 쿼리를 씁니다.
그래서 `include_types` 화이트리스트로 **볼 수 있는 라벨을 제한** 합니다.

In [23]:
# Text2Cypher 체인 구성 — 스키마 화이트리스트가 핵심이다
uv_install(['langchain-neo4j'])

cypher_chain, neo4j_graph = None, None
if NEO4J_READY:
    cypher_chain, neo4j_graph = gstack.build_cypher_qa_chain(
        llm, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, verbose=False)

    print(f"\nDB 전체 스키마: {len(neo4j_graph.schema):,}자 "
          "— 4장이 만든 entity/Chunk/__Node__ 까지 전부 들어 있다")
    print("→ include_types 로 걸러 LLM 에게는 이만큼만 보여 준다:\n")
    gstack.print_chain_schema(cypher_chain)
    print(f"\n(필터 후 {len(cypher_chain.graph_schema):,}자)")
else:
    print("Neo4j 미연결 — 7장을 건너뜁니다")

[uv] 설치 완료: ['langchain-neo4j']



DB 전체 스키마: 46,548자 — 4장이 만든 entity/Chunk/__Node__ 까지 전부 들어 있다
→ include_types 로 걸러 LLM 에게는 이만큼만 보여 준다:

Node properties:
Document {text: STRING, doc_id: STRING, embedding: LIST}
Topic {name: STRING}
Relationship properties:

The relationships:
(:Document)-[:MENTIONS]->(:Topic)
(:Topic)-[:PART_OF]->(:Topic)
(:Topic)-[:NEEDS]->(:Topic)
(:Topic)-[:CONSTRAINS]->(:Topic)
(:Topic)-[:DEFINES]->(:Topic)
(:Topic)-[:REPORTS]->(:Topic)

(필터 후 324자)


In [24]:
# 자연어 질문 → 생성된 Cypher → 조회 결과 → 답변
if cypher_chain:
    for question in ["프로젝트 태그를 가진 문서는 몇 개인가?",
                     "가장 많은 문서가 언급한 태그 3개는?",
                     "'권한' 태그에서 2홉 안에 닿는 태그는 무엇인가?"]:
        gstack.ask_cypher(cypher_chain, question)


질문: 프로젝트 태그를 가진 문서는 몇 개인가?


  생성된 Cypher: MATCH (d:Document)-[:MENTIONS]->(t:Topic {name: '프로젝트'}) RETURN COUNT(DISTINCT d) AS documentCount
  조회 결과 1행: [{'documentCount': 2}]
  답변: 2개

질문: 가장 많은 문서가 언급한 태그 3개는?


  생성된 Cypher: MATCH (d:Document)-[:MENTIONS]->(t:Topic) RETURN t.name AS tag, count(DISTINCT d) AS docCount ORDER BY docCount DESC LIMIT 3
  조회 결과 3행: [{'tag': 'RAG', 'docCount': 3}, {'tag': '보안', 'docCount': 2}, {'tag': '프로젝트', 'docCount': 2}]
  답변: 가장 많이 언급된 태그는 RAG(3개), 보안(2개), 프로젝트(2개)입니다.

질문: '권한' 태그에서 2홉 안에 닿는 태그는 무엇인가?


  생성된 Cypher: MATCH (t:Topic {name: '권한'})-[:PART_OF|NEEDS|CONSTRAINS|DEFINES|REPORTS*1..2]->(other:Topic) RETURN DISTINCT other.name AS tag
  조회 결과 2행: [{'tag': '보안'}, {'tag': '프로젝트'}]
  답변: 권한 태그에서 2홉 안에 닿는 태그는 보안과 프로젝트입니다.


### 7-1. Text2Cypher 를 쓸 때 챙길 것

**세 검색 방식은 잘하는 질문이 다릅니다**

| 질문 유형 | 잘하는 방식 |
|---|---|
| "무릎 아플 때 어떻게 했지?" (내용 조회) | **Vector RAG** (3장) |
| "회의록과 보안 지침은 어떻게 이어지나?" (관계 추론) | **GraphRAG** (4장) |
| "프로젝트 태그 문서가 **몇 개**냐", "상위 3개" (집계·정확 조회) | **Text2Cypher** (7장) |

**보안** — 이 체인은 LLM 이 만든 쿼리를 그대로 실행하므로 LangChain 이
`allow_dangerous_requests=True` 로 명시적 동의를 요구합니다. 실습은 로컬 컨테이너라 괜찮지만
운영에서는 이렇게 방어합니다.

1. **읽기 전용 계정** 으로 접속한다 (가장 확실한 방어 — 프롬프트가 뚫려도 쓰기가 안 된다)
2. 프롬프트에 `CREATE/MERGE/DELETE/SET 금지` 규칙을 넣는다 (라이브러리에 이미 포함)
3. `include_types` 로 접근 가능한 라벨을 제한한다
4. `validate_cypher=True` 로 관계 방향 오류를 자동 교정한다

**한계** — `include_types` 는 **라벨과 관계 타입만** 거릅니다. 속성은 못 거르므로
`Document.embedding`(384차원 리스트) 같은 것이 스키마에 그대로 남습니다.
그래서 라이브러리의 Cypher 프롬프트에 *"embedding 속성은 절대 RETURN 하지 마세요"* 규칙을
넣어 두었습니다 — 프롬프트로 막아야 하는 영역이 남는다는 점을 기억하세요.

### 7-2. 실패를 프롬프트 규칙으로 옮기기

이 노트북을 만들며 **실제로 겪은** 실패들입니다. 모두 모델을 바꿔서가 아니라
프롬프트에 규칙을 더해서 고쳤습니다.

| 증상 | LLM 이 생성한 것 | 넣은 규칙 |
|---|---|---|
| `SyntaxError` (가변 길이) | `[:PART_OF\|:CONSTRAINS*1..2]` | 콜론은 **한 번만** |
| `SyntaxError` (가변 길이) | `[:PART_OF\|CONSTRAINS]*1..2` | `*1..2` 는 **대괄호 안** 관계 목록 뒤 |
| 조회 결과 0건 | `{name: 'project'}` — 한국어 값을 **번역** | 질문의 값은 **원문 그대로** |
| `SyntaxError: Invalid input 'We'` | 쿼리 앞에 영어 설명 문장을 붙임 | 출력의 **첫 단어는 반드시 `MATCH`** |
| 답변이 `I don't know` / 영어 | (조회는 성공했는데 답변이 부정) | 기본 QA 프롬프트가 영어 → **한국어 QA 프롬프트** 로 교체 |

**그래도 100% 는 아닙니다.** `openrouter/free` 처럼 요청마다 모델이 바뀌는 설정에서
같은 질문을 4번 돌려 본 결과 **3번 성공 / 1번 실패** 였습니다. 실패한 회차의 원인도
매번 달랐습니다. 그래서 두 가지가 필요합니다.

1. **실패해도 죽지 않게** — `ask_cypher()` 는 예외를 잡아 오류를 보여 주고 다음 질문으로 넘어갑니다.
   (위 셀에서 한 질문이 실패해도 나머지가 실행되는 이유입니다.)
2. **모델을 고정** — 실습 결과를 안정적으로 보려면 `.env` 의 `LLM_PROVIDER` 를
   `ollama`(qwen3:8b)나 `google` 처럼 모델이 고정된 공급자로 바꾸세요.

> **Text2Cypher 운영은 결국 "실패 유형을 모아 프롬프트 규칙으로 옮기는 일"** 입니다.
> 실패한 질문·생성된 쿼리·오류 메시지를 로그로 남기는 것이 첫걸음입니다.

> ⚠️ 프롬프트에 Cypher 예시를 넣을 때 주의: LangChain `PromptTemplate` 은 `{...}` 를 변수로
> 읽습니다. `{name: '프로젝트'}` 같은 예시는 `{{name: '프로젝트'}}` 로 이스케이프해야 합니다
> (안 하면 `Input to PromptTemplate is missing variables` 오류).

In [25]:
# 정리 — 연결 종료 (데이터는 컨테이너에 남는다)
if NEO4J_READY:
    store.close()
    print("Neo4j 벡터 스토어 연결 종료")

# 6장에서 연 Bolt 연결들(아직 안 열었으면 조용히 건너뛴다)
for extra in (globals().get("neo_graph"), globals().get("mem_graph")):
    if extra is not None and extra.connected:
        extra.close()
        print(f"{extra.label} 연결 종료")

print("""
컨테이너 정리가 필요하면 CMD 에서:
  docker stop neo4j memgraph       REM 중지(데이터 유지)
  docker rm -f neo4j memgraph      REM 삭제(데이터도 삭제)
""")

Neo4j 벡터 스토어 연결 종료
Neo4j 연결 종료
Memgraph 연결 종료

컨테이너 정리가 필요하면 CMD 에서:
  docker stop neo4j memgraph       REM 중지(데이터 유지)
  docker rm -f neo4j memgraph      REM 삭제(데이터도 삭제)



---
## 8. 정리

### 배운 것

| 절 | 내용 | 핵심 API |
|---|---|---|
| 1 | 지식 그래프 원리 · 방향(direction) · multi-hop 탐색 | `rag.KnowledgeGraph`, `query_neighbors()`, `multi_hop_query()` |
| 2 | Neo4j 벡터 인덱스 + 그래프 확장 검색 | `rag.Neo4jVectorStore`, `db.index.vector.queryNodes` |
| 3 | LlamaIndex RAG (+ 공급자 무관 어댑터) | `llama_rag.configure()`, `build_vector_index()` |
| 4 | GraphRAG — LLM 자동 트리플 추출 + 커뮤니티 요약 → 전역 질문 | `llama_rag.build_property_graph_index()`, `graph_stack.detect_communities()`/`global_answer()` |
| 5 | Hybrid RAG = 벡터 + 그래프 | `rag.HybridRAG` |
| 6 | 그래프 DB 갈아타기 — Memgraph(같은 Cypher) | `graph_stack.BoltGraph`, `benchmark_graph_db()` |
| 7 | 자연어 → Cypher (집계·정확 조회) | `graph_stack.build_cypher_qa_chain()`, `ask_cypher()` |

### 기억할 원칙

1. **그래프는 공짜가 아니다** — 관계를 사람이 설계하거나 LLM 이 추출해야 하고, 둘 다 비용이 든다.
2. **GraphRAG 품질 = 추출 LLM 품질** — 작은 모델로 뽑은 그래프는 잡음이 더 많다.
3. **Neo4j 하나로 벡터 + 그래프** — 5.11+ 벡터 인덱스로 저장소를 이원화하지 않아도 된다.
4. **프레임워크는 도구** — LangChain 은 조립이 보이고, LlamaIndex 는 빠르게 세운다.
   어댑터를 두면 어느 쪽이든 `.env` 한 줄로 공급자를 바꿀 수 있다.
5. **Vector 먼저, Graph 는 필요할 때** — 실패 질문의 유형을 보고 도입을 결정한다.
6. **DB 는 요구사항이 고른다** — 실시간 갱신이 잦으면 인메모리 엔진(Memgraph)이 유리하다.
   Cypher 호환이면 이사 비용이 거의 없다 — 접속 주소만 바꾸면 된다.
7. **질문 유형이 검색 방식을 고른다** — 내용 조회는 Vector, 관계 추론은 Graph,
   집계·정확 조회는 Text2Cypher, 전체 조망은 커뮤니티 요약.

### 다음 편

- [`M04_4_agent_memory.ipynb`](M04_4_agent_memory.ipynb) — 에이전트 메모리를 직접 구현해 RAG 와 결합 · Deep-Knowledge Agent
- [`M04_5_memory.ipynb`](M04_5_memory.ipynb) — 같은 메모리를 LangGraph 표준 부품으로(체크포인터 · Store · 영속화)